In [2]:
import torch 
import torch.nn as nn
import torchvision 
import torchvision.transforms as transforms

import matplotlib.pyplot as plt

import torch.backends
import torch.backends.mps

import torch.utils
import torch.utils.data

from torch.optim.lr_scheduler import StepLR
from torch.utils.tensorboard import SummaryWriter

In [3]:
# create 1d tensor
t = torch.arange(1, 11)  
t

tensor([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10])

In [4]:
unfolded = t.unfold(0, 3, 2) # apply on dimension 0, window size 3, stride (step) 2
unfolded

tensor([[1, 2, 3],
        [3, 4, 5],
        [5, 6, 7],
        [7, 8, 9]])

In [5]:
# 2d tensor 
image = torch.arange(1, 17).reshape(4, 4) # 4 x 4 tensor
image, image.shape

(tensor([[ 1,  2,  3,  4],
         [ 5,  6,  7,  8],
         [ 9, 10, 11, 12],
         [13, 14, 15, 16]]),
 torch.Size([4, 4]))

In [6]:
patches = image.unfold(0, 2, 1)  # col axis (here cols are first), window 2, step 1
patches, patches.shape

(tensor([[[ 1,  5],
          [ 2,  6],
          [ 3,  7],
          [ 4,  8]],
 
         [[ 5,  9],
          [ 6, 10],
          [ 7, 11],
          [ 8, 12]],
 
         [[ 9, 13],
          [10, 14],
          [11, 15],
          [12, 16]]]),
 torch.Size([3, 4, 2]))

In [7]:
patches = patches.unfold(1, 2, 1) # col axis 1, window 2, step 1
patches

tensor([[[[ 1,  2],
          [ 5,  6]],

         [[ 2,  3],
          [ 6,  7]],

         [[ 3,  4],
          [ 7,  8]]],


        [[[ 5,  6],
          [ 9, 10]],

         [[ 6,  7],
          [10, 11]],

         [[ 7,  8],
          [11, 12]]],


        [[[ 9, 10],
          [13, 14]],

         [[10, 11],
          [14, 15]],

         [[11, 12],
          [15, 16]]]])

In [8]:
patches.contiguous().view(1, -1, 2 * 2) # contiguous() is needed to reshape

tensor([[[ 1,  2,  5,  6],
         [ 2,  3,  6,  7],
         [ 3,  4,  7,  8],
         [ 5,  6,  9, 10],
         [ 6,  7, 10, 11],
         [ 7,  8, 11, 12],
         [ 9, 10, 13, 14],
         [10, 11, 14, 15],
         [11, 12, 15, 16]]])

In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiFCPatchProcessor(nn.Module):
    def __init__(self, input_channels, patch_size, num_layers, output_per_layer, num_classes, image_height, image_width):
        super(MultiFCPatchProcessor, self).__init__()
        self.input_channels = input_channels
        self.patch_size = patch_size
        self.num_layers = num_layers
        self.output_per_layer = output_per_layer
        self.num_classes = num_classes

        output_height = image_height - self.patch_size + 1
        output_width = image_width - self.patch_size + 1

        print(output_height, output_width)

        # Create multiple fully connected layers
        self.fc_layers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(input_channels * patch_size * patch_size, output_per_layer), 
                nn.LeakyReLU(negative_slope=0.01)
                ) for _ in range(num_layers)
        ])

        self.final_fc = nn.Sequential(
            nn.Linear(output_height * output_width * num_layers, num_classes), 
            nn.LeakyReLU(negative_slope=0.01)
        )

    def forward(self, x):
        # Calculate dimensions based on the patch size (assuming stride=1 and no padding for simplicity)
        output_height = x.shape[2] - self.patch_size + 1
        output_width = x.shape[3] - self.patch_size + 1

        # Extract patches
        patches = x.unfold(2, self.patch_size, 1).unfold(3, self.patch_size, 1)
        patches = patches.contiguous().view(x.size(0), -1, self.input_channels * self.patch_size * self.patch_size)

        # Process each patch through each FC layer
        outputs = []
        for fc in self.fc_layers:
            # Apply the fully connected layer to each patch
            out = fc(patches)
            # Reshape to match the spatial layout
            out = out.view(x.size(0), output_height, output_width, self.output_per_layer)

            print(f"Out shape: {out.shape}")

            outputs.append(out)

        print(f"Ouptuts shape: {len(outputs)}")

        # Combine outputs across all layers
        # Concatenate along the last dimension to combine features from different FC layers
        combined_output = torch.cat(outputs, dim=3)  # Change dimension if needed

        print(f"Combined output shape: {combined_output.shape}")

         # Concatenate outputs across all layers
        combined_output = torch.cat(outputs, dim=3)

        print(f"Combined output shape: {combined_output.shape}")

        # Flatten the output
        combined_output = combined_output.view(x.size(0), -1)

        print(f"Flattened output shape: {combined_output.shape}")

        # Pass through final fully connected layer
        final_output = self.final_fc(combined_output)

        print(f"final output shape: {final_output.shape}")

        return final_output

# Example usage
input_image = torch.randn(3, 3, 8, 8)  # Simulate a single 3-channel 8x8 image

model = MultiFCPatchProcessor(input_channels=3, patch_size=3, num_layers=10, output_per_layer=1, num_classes=10, image_height = input_image.shape[2], image_width=input_image.shape[3])
output_feature_map = model(input_image)

print(output_feature_map.shape)  # Expected shape: [1, 7, 7, 10] for 10 outputs at each of the 7x7 positions


6 6
Out shape: torch.Size([3, 6, 6, 1])
Out shape: torch.Size([3, 6, 6, 1])
Out shape: torch.Size([3, 6, 6, 1])
Out shape: torch.Size([3, 6, 6, 1])
Out shape: torch.Size([3, 6, 6, 1])
Out shape: torch.Size([3, 6, 6, 1])
Out shape: torch.Size([3, 6, 6, 1])
Out shape: torch.Size([3, 6, 6, 1])
Out shape: torch.Size([3, 6, 6, 1])
Out shape: torch.Size([3, 6, 6, 1])
Ouptuts shape: 10
Combined output shape: torch.Size([3, 6, 6, 10])
Combined output shape: torch.Size([3, 6, 6, 10])
Flattened output shape: torch.Size([3, 360])
final output shape: torch.Size([3, 10])
torch.Size([3, 10])


In [12]:
image = torch.arange(1, 17).reshape(4, 4) # 4 x 4 tensor

# reshape into 2 x 8 tensor
image.reshape(8, 2)

tensor([[ 1,  2],
        [ 3,  4],
        [ 5,  6],
        [ 7,  8],
        [ 9, 10],
        [11, 12],
        [13, 14],
        [15, 16]])